In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

# Use a model better suited for tool calling on Groq
model = ChatGroq(model="llama-3.3-70b-versatile")


In [63]:
from langchain_tavily import TavilySearchResults

tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general"
)

tavily_search_tool.invoke("What is the current AI News")

{'query': 'What is the current AI News',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/@AINewsOfficial',
   'title': 'AI News - YouTube',
   'content': 'AI News delivers the latest breakthroughs in artificial intelligence, machine learning, deep learning, brain computer interface, robotics, and other future',
   'score': 0.99957615,
   'raw_content': None},
  {'url': 'https://www.wsj.com/tech/ai?gaa_at=eafs&gaa_n=AWEtsqevq6aXsXNwEydn_ezRSdU4KRd6Fkliv9rCOSRhc0u5AZ0DgeIP55lm&gaa_ts=69d54a3c&gaa_sig=857DY2F1KnJhb9KnF6t_RntMhNZvC2IfqPBf5gMMEhGzD46swD-HFeTGBRdbMoTXDc77oJxwm6I7veLl-QFaqA%3D%3D',
   'title': 'Artificial Intelligence - Latest AI News and Analysis - WSJ.com',
   'content': 'The latest artificial intelligence news coverage focusing on the technology, tools and the companies building AI technology.',
   'score': 0.9990601,
   'raw_content': None},
  {'url': 'https://www.reuters.com/technology/artificial-intelligence/',

In [64]:
from langchain.tools import tool

@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

In [65]:
tool

<function langchain_core.tools.convert.tool(name_or_callable: str | collections.abc.Callable | None = None, runnable: langchain_core.runnables.base.Runnable | None = None, *args: Any, description: str | None = None, return_direct: bool = False, args_schema: type[pydantic.main.BaseModel] | dict[str, Any] | None = None, infer_schema: bool = True, response_format: Literal['content', 'content_and_artifact'] = 'content', parse_docstring: bool = False, error_on_invalid_docstring: bool = True, extras: dict[str, Any] | None = None) -> langchain_core.tools.base.BaseTool | collections.abc.Callable[[collections.abc.Callable | langchain_core.runnables.base.Runnable], langchain_core.tools.base.BaseTool]>

In [66]:
# from langchain.agents import create_agent

# agents=create_agent(
#     model=model,
#     tools=[tavily_search_tool, calc]
# )

# or

from langchain.agents import create_agent

agents=create_agent(model, [tavily_search_tool, calc])

In [67]:
user_input="What is the current AI news of Anthorpic and then calculate 5 + 5"

for step in agents.stream(
    {"messages": user_input},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()
    # print(step["messages"][-1])

================================ Human Message =================================

What is the current AI news of Anthorpic and then calculate 5 + 5
================================== Ai Message ==================================
Tool Calls:
  tavily_search (1qadcv5a0)
 Call ID: 1qadcv5a0
  Args:
    query: Anthorpic AI news
    topic: news
  calculator (rsq7tgqbm)
 Call ID: rsq7tgqbm
  Args:
    expression: 5 + 5
================================= Tool Message =================================
Name: calculator

10
================================== Ai Message ==================================

The current AI news of Anthorpic is about their new AI model, Mythos, which is said to be a step change in capabilities. The model is currently being tested with early access customers, and it's reported to be by far the most powerful AI model Anthropic has ever developed. Additionally, Anthropic has announced the launch of The Anthropic Institute, a new effort to confront the most significant ch

In [69]:
agents.invoke({"messages": [("user", "What happened at the last wimbledon")]})
# agents.invoke({"messages": [("user", "What is the current AI news of Anthorpic and then calculate 235/2")]})['messages'][2].content


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=tavily_search>{"query": "last Wimbledon", "topic": "news", "time_range": "day"}'}}